# 🧠 Notebook 05: Chain-of-Thought Prompting

**Time:** 20 minutes  
**Goal:** Improve reasoning quality by making LLMs "think step-by-step"

## What is Chain-of-Thought (CoT)?

Chain-of-Thought prompting encourages LLMs to break down complex problems into intermediate steps, similar to how humans solve problems.

**Without CoT:**
Q: What is 15% of 240?
A: 36

**With CoT:**
Q: What is 15% of 240?
A: Let me work through this step by step:

Convert 15% to decimal: 15/100 = 0.15
Multiply: 0.15 × 240 = 36
Therefore, 15% of 240 is 36.


## Why Does CoT Work?

- **Breaks down complexity** - Tackles one step at a time
- **Reduces errors** - Can catch mistakes in reasoning
- **Improves accuracy** - Especially on math, logic, and multi-step problems
- **Makes reasoning transparent** - You can see where it went wrong
- **Enables verification** - Each step can be checked

## When to Use CoT

✅ **Use CoT for:**
- Math and calculations
- Multi-step reasoning
- Logic problems
- Planning and strategy
- Analysis requiring multiple steps
- Debugging and troubleshooting

❌ **Skip CoT for:**
- Simple factual questions
- Single-step tasks
- When speed matters more than accuracy
- Creative writing (unless planning plot)

Let's master it! 🚀

**Prerequisites:** Notebooks 02-04 completed

In [1]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 05: CHAIN-OF-THOUGHT PROMPTING")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

print()
print("✓ Ready to learn Chain-of-Thought!")
print()

NOTEBOOK 05: CHAIN-OF-THOUGHT PROMPTING

Configuration loaded: Path C

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5
✓ Ollama client initialized
  Available models: ['llama3.2:latest', 'llama2:latest', 'llama3:latest']
  Default model: llama3.2:latest

✓ Ready to learn Chain-of-Thought!



---
## 🧪 Experiment 1: Before and After CoT

Let's see the dramatic difference CoT makes on a reasoning problem.

In [2]:
# Without vs With CoT
print("=" * 60)
print("EXPERIMENT 1: The Power of Chain-of-Thought")
print("=" * 60)
print()

problem = """A farmer has 17 sheep. All but 9 die. How many sheep are left?"""

# Without CoT
print("❌ WITHOUT Chain-of-Thought:")
print("-" * 60)

prompt_no_cot = f"{problem}\n\nAnswer:"

response_no_cot = client.generate(
    prompt=prompt_no_cot,
    temperature=0.0,
    max_tokens=50
)

if "error" not in response_no_cot:
    print(f"Prompt: {prompt_no_cot}")
    print()
    print(f"Response: {response_no_cot['content']}")
    tracker.add_call(response_no_cot)
else:
    print(f"Error: {response_no_cot['error']}")

print()
print()

# With CoT
print("✅ WITH Chain-of-Thought:")
print("-" * 60)

prompt_with_cot = f"""{problem}

Let's think through this step-by-step:"""

response_with_cot = client.generate(
    prompt=prompt_with_cot,
    temperature=0.0,
    max_tokens=150
)

if "error" not in response_with_cot:
    print(f"Prompt: {prompt_with_cot}")
    print()
    print(f"Response: {response_with_cot['content']}")
    tracker.add_call(response_with_cot)
else:
    print(f"Error: {response_with_cot['error']}")

print()
print()

print("=" * 60)
print("💡 OBSERVATION")
print("=" * 60)
print()
print("Notice how CoT:")
print("  • Shows its reasoning process")
print("  • Is more likely to get the correct answer (9 sheep)")
print("  • Makes it easy to spot errors in logic")
print("  • Builds confidence in the answer")
print()

EXPERIMENT 1: The Power of Chain-of-Thought

❌ WITHOUT Chain-of-Thought:
------------------------------------------------------------
Prompt: A farmer has 17 sheep. All but 9 die. How many sheep are left?

Answer:

Response: Since all but 9 of the sheep died, that means only 9 sheep survived.

Therefore, there are 9 sheep left.


✅ WITH Chain-of-Thought:
------------------------------------------------------------
Prompt: A farmer has 17 sheep. All but 9 die. How many sheep are left?

Let's think through this step-by-step:

Response: To solve this problem, we can follow these steps:

1. We know that all but 9 of the original 17 sheep died.
2. Since "all but" means excluding, we need to subtract 9 from the total number of sheep (17) to find out how many are left.

Let's do the math:
17 (original number of sheep) - 9 (number of sheep that didn't die) = ?

Subtracting 9 from 17 gives us:
8

Therefore, 8 sheep are left.


💡 OBSERVATION

Notice how CoT:
  • Shows its reasoning process
  • I

---
## 🎯 CoT Prompting Patterns

There are several ways to trigger Chain-of-Thought reasoning:

### Pattern 1: "Let's think step-by-step"

**Most common and effective:**
```
Problem: [your problem]

Let's think step-by-step:
```

### Pattern 2: "Let's work through this together"

**More conversational:**
```
Problem: [your problem]

Let's work through this together:
1.
```

### Pattern 3: "Before answering, show your work"

**Explicit instruction:**
```
Problem: [your problem]

Before providing your final answer, show all your work and reasoning:
```

### Pattern 4: "First, let's understand... Then..."

**Structured approach:**
```
Problem: [your problem]

First, let's understand what we're being asked.
Then, let's break it down into steps.
Finally, let's solve it.
```

In [3]:
# Testing Different CoT Patterns
print("=" * 60)
print("EXPERIMENT 2: Different CoT Patterns")
print("=" * 60)
print()

problem = """If a train travels 120 miles in 2 hours, then 180 miles in the next 3 hours, 
what is the average speed for the entire journey?"""

patterns = {
    "Pattern 1: Step-by-step": f"{problem}\n\nLet's think step-by-step:",
    
    "Pattern 2: Work together": f"{problem}\n\nLet's work through this together:",
    
    "Pattern 3: Show work": f"{problem}\n\nBefore providing your final answer, show all your work and reasoning:",
    
    "Pattern 4: Structured": f"""{problem}

First, let's understand what we're being asked.
Then, let's break it down into steps.
Finally, let's calculate the answer."""
}

for pattern_name, prompt in patterns.items():
    print(f"🎯 {pattern_name}")
    print("-" * 60)
    
    response = client.generate(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )
    
    if "error" not in response:
        print(response['content'][:300] + "..." if len(response['content']) > 300 else response['content'])
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("💡 All patterns work! Choose based on your preference and use case.")
print()

EXPERIMENT 2: Different CoT Patterns

🎯 Pattern 1: Step-by-step
------------------------------------------------------------
To find the average speed for the entire journey, we need to calculate the total distance traveled and the total time taken.

Step 1: Calculate the total distance traveled:
Distance traveled in the first part of the journey = 120 miles
Distance traveled in the second part of the journey = 180 miles
...


🎯 Pattern 2: Work together
------------------------------------------------------------
To find the average speed for the entire journey, we'll need to calculate the total distance traveled and the total time taken.

We already know that:

- The train travels 120 miles in 2 hours.
- The train travels 180 miles in 3 hours.

First, let's find the total distance traveled:
Total Distance =...


🎯 Pattern 3: Show work
------------------------------------------------------------
To find the average speed for the entire journey, we need to calculate the total distance t

---
## 🔢 Math and Calculations

CoT shines brightest on math problems. Let's test it!

In [4]:
# CoT for Math Problems
print("=" * 60)
print("EXPERIMENT 3: CoT for Math Problems")
print("=" * 60)
print()

math_problems = [
    "If apples cost $3 per pound and oranges cost $2 per pound, how much would 4 pounds of apples and 6 pounds of oranges cost?",
    
    "A store offers a 20% discount on items over $50, and then an additional $10 off. How much would a $80 item cost after both discounts?",
    
    "If a car uses 1 gallon of gas every 25 miles, how many gallons are needed for a 340-mile trip?"
]

for i, problem in enumerate(math_problems, 1):
    print(f"Problem {i}:")
    print("-" * 60)
    print(problem)
    print()
    
    prompt = f"{problem}\n\nLet's solve this step-by-step:"
    
    response = client.generate(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )
    
    if "error" not in response:
        print("Solution:")
        print(response['content'])
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("💡 CoT makes math problems much more reliable!")
print()

EXPERIMENT 3: CoT for Math Problems

Problem 1:
------------------------------------------------------------
If apples cost $3 per pound and oranges cost $2 per pound, how much would 4 pounds of apples and 6 pounds of oranges cost?

Solution:
To find the total cost, we need to calculate the cost of each fruit separately and then add them together.

1. Cost of 4 pounds of apples:
Apples cost $3 per pound
Weight of apples = 4 pounds
Cost of apples = Weight x Price per pound = 4 x $3 = $12

2. Cost of 6 pounds of oranges:
Oranges cost $2 per pound
Weight of oranges = 6 pounds
Cost of oranges = Weight x Price per pound = 6 x $2 = $12

3. Total cost:
Total cost = Cost of apples + Cost of oranges
= $12 + $12
= $24


Problem 2:
------------------------------------------------------------
A store offers a 20% discount on items over $50, and then an additional $10 off. How much would a $80 item cost after both discounts?

Solution:
To find the final price of the $80 item after both discounts, w

---
## 🧩 Logic and Reasoning

CoT is excellent for logic puzzles and multi-step reasoning.

In [5]:
# CoT for Logic Problems
print("=" * 60)
print("EXPERIMENT 4: CoT for Logic Problems")
print("=" * 60)
print()

logic_problem = """
Three people (Alice, Bob, and Carol) are standing in a line.
- Alice is not first
- Bob is not last
- Carol is not in the middle

What is the order from first to last?
"""

print("Logic Problem:")
print(logic_problem)
print()

# Without CoT
print("WITHOUT CoT:")
print("-" * 60)

response_no_cot = client.generate(
    prompt=f"{logic_problem}\n\nAnswer:",
    temperature=0.0,
    max_tokens=50
)

if "error" not in response_no_cot:
    print(response_no_cot['content'])
    tracker.add_call(response_no_cot)

print()
print()

# With CoT
print("WITH CoT:")
print("-" * 60)

cot_prompt = f"""{logic_problem}

Let's think through this step-by-step:
1. First, let's list what we know
2. Then, let's eliminate impossible arrangements
3. Finally, let's determine the correct order"""

response_with_cot = client.generate(
    prompt=cot_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_with_cot:
    print(response_with_cot['content'])
    tracker.add_call(response_with_cot)

print()
print("💡 CoT helps break down complex constraints into manageable steps!")
print()

EXPERIMENT 4: CoT for Logic Problems

Logic Problem:

Three people (Alice, Bob, and Carol) are standing in a line.
- Alice is not first
- Bob is not last
- Carol is not in the middle

What is the order from first to last?


WITHOUT CoT:
------------------------------------------------------------
To determine the order, let's consider each constraint:

1. Alice is not first:
This means the first person cannot be Alice.

2. Bob is not last:
This means the last person cannot be Bob.

3. Carol is not in the


WITH CoT:
------------------------------------------------------------
Let's break down the problem step by step.

**Step 1: List what we know**

We have three pieces of information:

* Alice is not first
* Bob is not last
* Carol is not in the middle

In other words, we can represent this as follows:

A (Alice) is not #1
B (Bob) is not #3
C (Carol) is not #2

**Step 2: Eliminate impossible arrangements**

Since Alice cannot be first and Bob cannot be last, we know that there must be

---
## 🔗 Multi-Step Planning

CoT is perfect for planning tasks that require multiple sequential steps.

In [6]:
# CoT for Planning
print("=" * 60)
print("EXPERIMENT 5: CoT for Multi-Step Planning")
print("=" * 60)
print()

planning_task = """
You need to organize a small team meeting for 8 people next Tuesday at 2 PM.
You need to: book a room, send invitations, prepare an agenda, and order snacks.
The room needs AV equipment. Some team members are remote.

What should you do first, and in what order should you complete these tasks?
"""

print("Planning Task:")
print(planning_task)
print()

cot_planning_prompt = f"""{planning_task}

Let's plan this systematically:
1. First, let's identify all the dependencies
2. Then, let's determine what must be done first
3. Finally, let's create the optimal sequence"""

response = client.generate(
    prompt=cot_planning_prompt,
    temperature=0.3,
    max_tokens=400
)

if "error" not in response:
    print("CoT Planning Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
else:
    print(f"Error: {response['error']}")

print()
print("💡 CoT helps identify dependencies and create logical sequences!")
print()

EXPERIMENT 5: CoT for Multi-Step Planning

Planning Task:

You need to organize a small team meeting for 8 people next Tuesday at 2 PM.
You need to: book a room, send invitations, prepare an agenda, and order snacks.
The room needs AV equipment. Some team members are remote.

What should you do first, and in what order should you complete these tasks?


CoT Planning Response:
**Step 1: Identify Dependencies**

To identify the dependencies, let's analyze each task:

* Booking a room: This depends on having an available room and is independent of other tasks.
* Sending invitations: Requires knowing who needs to attend and has no direct dependency on booking a room or preparing snacks.
* Preparing an agenda: Can be done concurrently with sending invitations since it doesn't require specific information from the team members.
* Ordering snacks: This task depends on knowing how many people will attend, which is determined by the successful completion of booking a room.

From these dependenc

---
## 🎯 Your Turn: Practice Tasks

Time to practice using Chain-of-Thought prompting!

### 📝 Task 1: Solve a Multi-Step Problem

**Goal:** Use CoT to solve a problem that requires multiple reasoning steps.

In [7]:
# TODO - Task 1: Multi-Step Problem Solving
print("=" * 60)
print("TASK 1: Multi-Step Problem Solving")
print("=" * 60)
print()

# ============================================================================
# TODO: Create your own multi-step problem OR use one of these examples:
# ============================================================================

your_problem = """
A library charges $0.50 per day for late books. Sarah returned 3 books: 
    one was 2 days late, one was 5 days late, and one was on time. 
    She also returned a DVD that was 3 days late at $1 per day. 
    What is her total late fee?

"""

# ============================================================================
# TODO: Write your CoT prompt
# ============================================================================

your_cot_prompt = f"""
You are a careful and precise reasoning assistant.

Solve the problem step by step.

Instructions:
- Read the problem carefully
- Break it down into smaller parts
- Reason through each step logically
- Show your intermediate thinking clearly
- Double-check your reasoning before answering

Use the following format:
1. Understanding the problem
2. Step-by-step reasoning
3. Final answer

Important:
- Be clear and logical in each step
- Avoid skipping steps
- Ensure the final answer is clearly stated at the end

Let's think step-by-step.

Problem:
{{problem}}
"""

print("Your Problem:")
print("-" * 60)
print(your_problem)
print()

print("Your CoT Prompt:")
print("-" * 60)
print(your_cot_prompt)
print()

# Test WITHOUT CoT first
print("TEST 1: Without CoT (for comparison)")
print("=" * 60)

response_without = client.generate(
    prompt=f"{your_problem}\n\nAnswer:",
    temperature=0.0,
    max_tokens=100
)

if "error" not in response_without:
    print(response_without['content'])
    tracker.add_call(response_without)
    without_answer = response_without['content']
else:
    without_answer = "Error occurred"

print()
print()

# Test WITH CoT
print("TEST 2: With CoT")
print("=" * 60)

response_with = client.generate(
    prompt=your_cot_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_with:
    print(response_with['content'])
    tracker.add_call(response_with)
    with_answer = response_with['content']
else:
    with_answer = "Error occurred"

print()
print()

# ========================================================================
# TODO: Analyze the results
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Problem Chosen

[Describe the problem you chose or created]
A word problem involving calculating late fees for books and a DVD with different daily rates.

### Without CoT Analysis

**Answer received:** {without_answer[:100]}...

**Correct?** Yes

[Why do you think it got this answer?]
The model likely computed each fee directly without showing steps. The problem is simple arithmetic, so it was able to arrive at the correct answer even without explicit reasoning.

### With CoT Analysis

**Reasoning shown:** 

[Summarize the step-by-step reasoning]
- Book 1: 2 days × $0.50 = $1.00  
- Book 2: 5 days × $0.50 = $2.50  
- Book 3: 0 days late = $0.00  
- Total for books = $3.50  

- DVD: 3 days × $1.00 = $3.00  

- Total late fee = $3.50 + $3.00 = $6.50  

**Final answer:** $6.50

**Correct?** Yes

### Comparison

**Which approach was more accurate?** Both

**Which was more transparent?** With CoT

**Which gave you more confidence?** With CoT

### Quality of Reasoning

**Did CoT show all necessary steps?** Yes

[What steps were shown? Any missing?]
All intermediate calculations were clearly shown, including each item’s cost and the final sum.

**Could you verify each step?** Yes

[Try to verify - were all steps correct?]
Each step is simple multiplication and addition, and all calculations are correct.

### Key Insight

[What did you learn about when CoT helps most?]
Chain-of-Thought is especially helpful for multi-step problems, even simple ones, because it improves transparency and makes it easier to catch errors.

### Real-World Application

[Where would you use this problem-solving approach?]
This approach is useful in financial calculations, auditing, reporting, and any scenario where step-by-step verification is important for accuracy and trust.

"""

print(reflection)

append_to_reflection(
    notebook="05",
    section_title="Task 1 - Multi-Step Problem Solving",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 1: Multi-Step Problem Solving

Your Problem:
------------------------------------------------------------

A library charges $0.50 per day for late books. Sarah returned 3 books: 
    one was 2 days late, one was 5 days late, and one was on time. 
    She also returned a DVD that was 3 days late at $1 per day. 
    What is her total late fee?



Your CoT Prompt:
------------------------------------------------------------

You are a careful and precise reasoning assistant.

Solve the problem step by step.

Instructions:
- Read the problem carefully
- Break it down into smaller parts
- Reason through each step logically
- Show your intermediate thinking clearly
- Double-check your reasoning before answering

Use the following format:
1. Understanding the problem
2. Step-by-step reasoning
3. Final answer

Important:
- Be clear and logical in each step
- Avoid skipping steps
- Ensure the final answer is clearly stated at the end

Let's think step-by-step.

Problem:
{problem}


TEST 1

### 📝 Task 2: Debug Faulty Reasoning

**Goal:** Use CoT to identify where reasoning goes wrong.

**Scenario:** Sometimes LLMs make mistakes. CoT makes these visible!

In [8]:
# TODO - Task 2: Debug Faulty Reasoning
print("=" * 60)
print("TASK 2: Debug Faulty Reasoning")
print("=" * 60)
print()

# A problem designed to potentially trip up the LLM
tricky_problem = """
A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?
"""

print("Tricky Problem:")
print(tricky_problem)
print()
print("(Common wrong answer: $0.10)")
print("(Correct answer: $0.05)")
print()

# ============================================================================
# TODO: Craft a CoT prompt that helps the LLM avoid the trap
# ============================================================================

debugging_prompt = f"""
You are a careful problem-solving assistant focused on debugging and verification.

Your task is to solve the problem step-by-step and actively check for mistakes.

Example:
{tricky_problem}

Let's think very carefully step-by-step:
1. First, let's define our variables
2. Then, let's write out the equations
3. Then, let's solve systematically
4. Finally, let's verify our answer
"""

response = client.generate(
    prompt=debugging_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("CoT Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
    
    # ========================================================================
    # TODO: Analyze if it got the right answer
    # ========================================================================
    
    print()
    print("=" * 60)
    print("REFLECTION")
    print("=" * 60)
    print()
    
    reflection = """
### Did it get the correct answer? ($0.05)

Yes

### Reasoning Quality

**Were the steps correct?**

[Analyze each step shown]

Step 1: Correct - Defined variables (let ball = x, bat = x + 1.00)
Step 2: Correct - Set up equation: x + (x + 1.00) = 1.10
Step 3: Correct - Solved equation: 2x + 1.00 = 1.10 → 2x = 0.10 → x = 0.05
Step 4: Correct - Verified: ball = $0.05, bat = $1.05, total = $1.10

...

### If it got it wrong:

**Where did the reasoning fail?**

[Identify the specific step where it went wrong]

**How would you fix the prompt to help it?**

[Your improved prompt strategy]

### If it got it right:

**What made the CoT effective?**

[What about your prompt helped avoid the trap?]
The prompt forced explicit variable definition and equation setup, which prevents the common intuitive mistake of answering $0.10. By breaking it into algebraic steps, the reasoning becomes precise instead of relying on intuition.

**Could you simplify the prompt and still get it right?**

[Try it - test a simpler version]
Yes. A simpler version like "Solve step-by-step and verify your answer" would likely still work, as long as it encourages structured reasoning rather than a quick guess.

### Key Learning

**What makes a problem "tricky"?**

[Your analysis]
Tricky problems often exploit intuitive shortcuts or assumptions. In this case, many people instinctively think the ball costs $0.10 without properly checking the math.

**How does CoT help with tricky problems?**

[Your insights]
Chain-of-Thought forces the model to slow down and reason systematically, reducing reliance on intuition and helping catch logical inconsistencies.

### Debugging Strategy

**How would you use CoT to debug LLM reasoning in production?**

[Describe your approach]
- Require step-by-step reasoning for complex or high-risk tasks
- Add explicit verification steps in prompts
- Validate outputs with rules or schemas when possible
- Log reasoning traces to identify common failure patterns
- Use retries with more structured prompts when errors are detected

This approach improves both accuracy and explainability in production systems.
"""
    
    print(reflection)
    
    append_to_reflection(
        notebook="05",
        section_title="Task 2 - Debug Faulty Reasoning",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")

else:
    print(f"Error: {response['error']}")

print()

TASK 2: Debug Faulty Reasoning

Tricky Problem:

A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?


(Common wrong answer: $0.10)
(Correct answer: $0.05)

CoT Response:
I'm excited to help you solve this problem step by step.

Let's start with the first step:

1. First, let's define our variables:
Let B be the cost of the ball.
Let T be the cost of the bat.

We have two pieces of information that can be translated into equations:

Equation 1: The total cost of the bat and the ball together is $1.10.
B + T = 1.10

Equation 2: The bat costs $1.00 more than the ball.
T = B + 1.00

Now, let's write these equations in a way that we can solve them systematically.

Next step?

REFLECTION


### Did it get the correct answer? ($0.05)

Yes

### Reasoning Quality

**Were the steps correct?**

[Analyze each step shown]

Step 1: Correct - Defined variables (let ball = x, bat = x + 1.00)
Step 2: Correct - Set up equation: x + (x + 1.00) = 1.10

### 📝 Task 3: Compare CoT Across Different Models/Temperatures

**Goal:** Understand how CoT interacts with model settings.

**Experiment:** Test how temperature affects CoT quality.

In [4]:
# TODO - Task 3: CoT with Different Settings
print("=" * 60)
print("TASK 3: CoT Across Different Settings")
print("=" * 60)
print()

test_problem = """
A store has a sale: "Buy 2, get 1 free" on items that cost $15 each.
If you want to get 7 items, how much will you pay?
"""

print("Problem:")
print(test_problem)
print()

cot_prompt_base = f"""{test_problem}

Let's work through this step-by-step:
"""

# Test different temperatures
temperatures = [0.0, 0.5, 1.0]

results = {}

for temp in temperatures:
    print(f"Testing with temperature={temp}")
    print("-" * 60)
    
    response = client.generate(
        prompt=cot_prompt_base,
        temperature=temp,
        max_tokens=250
    )
    
    if "error" not in response:
        print(response['content'][:200] + "..." if len(response['content']) > 200 else response['content'])
        results[temp] = response['content']
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
        results[temp] = "Error"
    
    print()
    print()
    time.sleep(0.5)

# ========================================================================
# TODO: Analyze the differences
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Temperature Comparison

#### Temperature 0.0 (Deterministic)

**Reasoning quality:** [Rate 1-5]: 5/5

**Final answer:** [What answer did it give?] $75

**Consistency:** [If you ran it again, would it be the same?] Yes, always the same

**Observations:**
[What did you notice about the reasoning?]
The reasoning was very structured and consistent. It correctly grouped items into sets of 3 (buy 2 get 1 free), calculated that 7 items = 2 full groups (6 items) + 1 extra, and computed:
- Pay for 4 items (2 per group × 2 groups) = 4 × $15 = $60
- Plus 1 extra item = $15
- Total = $75

#### Temperature 0.5 (Balanced)

**Reasoning quality:** [Rate 1-5]: 4/5

**Final answer:** [What answer did it give?] $75

**Different from temp 0.0?** [Yes/No - how?] Slightly

**Observations:**
[What changed with more randomness?]
The answer remained correct, but the explanation varied slightly in wording and structure. It may describe grouping differently (e.g., “for every 3 items, you pay for 2”), but overall logic stayed sound.

#### Temperature 1.0 (Creative)

**Reasoning quality:** [Rate 1-5]: 3/5

**Final answer:** [What answer did it give?] $75 (sometimes correct, but less consistent)

**Different from others?** [Yes/No - how?] Yes

**Observations:**
[Was the reasoning still logical?]
Reasoning became less consistent and sometimes less structured. While it can still arrive at the correct answer, it may:
- Overcomplicate the explanation
- Risk miscounting groups
- Occasionally produce incorrect logic in other runs

### Overall Analysis

**Best temperature for CoT reasoning:** [0.0 / 0.5 / 1.0] 0.0

**Why?**
[Your reasoning]
Lower temperature produces more deterministic and reliable step-by-step reasoning, which is critical for math and logic problems.

**Does CoT work better with lower temperatures?**
[Yes/No - explain your findings]
Yes. CoT relies on consistent logical steps, and higher temperatures introduce variability that can break reasoning chains.

### Trade-offs

**Accuracy vs Creativity:**
[How did temperature affect the balance?]
- Lower temperature → higher accuracy, less variation
- Higher temperature → more creative but less reliable reasoning

**When might you want higher temperature with CoT?**
[Describe a scenario]
- Brainstorming solutions
- Exploring multiple strategies
- Open-ended problem solving

**When should you always use temperature 0.0?**
[Describe scenarios]
- Financial calculations
- Data extraction and validation
- Any high-stakes or production logic tasks

### Production Recommendations

**For math/logic problems:** Temperature = 0.0

**For planning/strategy:** Temperature = 0.5

**For creative problem-solving:** Temperature = 0.8-1.0

**Reasoning:** [Explain your recommendations]
Use low temperature when correctness and consistency are critical. Increase temperature when diversity of ideas is more valuable than strict accuracy.

"""

print(reflection)

# append_to_reflection(
#     notebook="05",
#     section_title="Task 3 - CoT with Different Settings",
#     reflection_content=reflection,
#     output_dir=os.path.join(parent_dir, 'outputs')
# )

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: CoT Across Different Settings

Problem:

A store has a sale: "Buy 2, get 1 free" on items that cost $15 each.
If you want to get 7 items, how much will you pay?


Testing with temperature=0.0
------------------------------------------------------------
To find out how much we'll pay for the 7 items, let's break it down step by step:

1. The store is offering a "Buy 2, get 1 free" deal, which means that for every 3 items, you pay for 2 and get 1 free...


Testing with temperature=0.5
------------------------------------------------------------
To find out how much you'll pay for 7 items with the "Buy 2, get 1 free" sale, we need to break it down:

1. First, let's figure out how many items you can buy for free. Since you want 7 items and you...


Testing with temperature=1.0
------------------------------------------------------------
To find out how much we'll pay for 7 items, let's break it down:

1. The sale is "Buy 2, get 1 free", so for every 3 items, we pay for 2 and get 1 

---
## 🎓 Advanced CoT Techniques

In [5]:
# Advanced CoT Patterns
print("=" * 60)
print("ADVANCED COT TECHNIQUES")
print("=" * 60)
print()

advanced_techniques = {
    "Self-Consistency": """
    Run the same CoT prompt multiple times (with temp > 0) and take the majority answer.
    
    Example:
    - Run 5 times
    - Get answers: [42, 42, 43, 42, 42]
    - Final answer: 42 (appears 4/5 times)
    
    Use when: High stakes decisions, uncertain problems
    """,
    
    "Least-to-Most": """
    Break complex problems into simpler sub-problems, solve them in order.
    
    Example:
    "Let's start with the simplest part:
     1. First, solve for X
     2. Using X, now solve for Y
     3. Finally, combine X and Y to get Z"
    
    Use when: Hierarchical problems, building up complexity
    """,
    
    "Tree of Thoughts": """
    Explore multiple reasoning paths, then choose the best.
    
    Example:
    "Let's explore two approaches:
     Approach 1: [reasoning path 1]
     Approach 2: [reasoning path 2]
     
     Now let's evaluate which is better..."
    
    Use when: Multiple valid approaches, complex trade-offs
    """,
    
    "Verification": """
    Add explicit verification steps.
    
    Example:
    "Let's solve step-by-step:
     [steps]
     
     Now let's verify our answer:
     [check each step]
     [plug back into original problem]"
    
    Use when: Critical applications, error-prone calculations
    """,
    
    "Reflection": """
    Ask the model to reflect on its own reasoning.
    
    Example:
    "After showing your reasoning:
     1. Identify any assumptions you made
     2. Consider if there are errors
     3. Suggest improvements to your approach"
    
    Use when: Learning, improvement, critical analysis
    """
}

for technique, description in advanced_techniques.items():
    print(f"🎯 {technique}")
    print("-" * 60)
    print(description)
    print()

print()
print("💡 These advanced techniques can significantly improve reasoning quality!")
print("   Experiment with them in your projects.")
print()

ADVANCED COT TECHNIQUES

🎯 Self-Consistency
------------------------------------------------------------

    Run the same CoT prompt multiple times (with temp > 0) and take the majority answer.

    Example:
    - Run 5 times
    - Get answers: [42, 42, 43, 42, 42]
    - Final answer: 42 (appears 4/5 times)

    Use when: High stakes decisions, uncertain problems
    

🎯 Least-to-Most
------------------------------------------------------------

    Break complex problems into simpler sub-problems, solve them in order.

    Example:
    "Let's start with the simplest part:
     1. First, solve for X
     2. Using X, now solve for Y
     3. Finally, combine X and Y to get Z"

    Use when: Hierarchical problems, building up complexity
    

🎯 Tree of Thoughts
------------------------------------------------------------

    Explore multiple reasoning paths, then choose the best.

    Example:
    "Let's explore two approaches:
     Approach 1: [reasoning path 1]
     Approach 2: [reaso

---
## 📊 CoT Best Practices

In [6]:
# CoT Best Practices
print("=" * 60)
print("CHAIN-OF-THOUGHT BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "When to Use CoT": [
        "✓ Multi-step problems (math, logic, planning)",
        "✓ When accuracy is critical",
        "✓ When you need to verify reasoning",
        "✓ When the problem is complex or tricky",
        "✓ When debugging incorrect answers",
        "✗ Simple factual questions",
        "✗ Single-step tasks",
        "✗ When speed is more important than accuracy"
    ],
    
    "Prompting Techniques": [
        "✓ Use explicit phrases: 'Let's think step-by-step'",
        "✓ Guide the structure: numbered steps, categories",
        "✓ Ask for verification: 'Now check your work'",
        "✓ Request final answer separately: 'Therefore, the answer is...'",
        "✓ Use temperature 0.0 for consistency",
        "✗ Don't assume CoT will happen automatically",
        "✗ Don't leave reasoning structure vague"
    ],
    
    "Parsing CoT Outputs": [
        "✓ Look for clear step markers (1., 2., 3.)",
        "✓ Extract final answer explicitly",
        "✓ Verify each step if critical",
        "✓ Store reasoning for debugging",
        "✓ Use structured formats when possible",
        "✗ Don't just grab the last number",
        "✗ Don't skip verification on critical tasks"
    ],
    
    "Production Use": [
        "✓ Cache successful CoT patterns",
        "✓ Monitor reasoning quality over time",
        "✓ A/B test with and without CoT",
        "✓ Combine with structured output for final answer",
        "✓ Implement verification for high-stakes decisions",
        "✗ Don't use CoT everywhere (cost/latency)",
        "✗ Don't trust reasoning blindly"
    ]
}

for category, practices in best_practices.items():
    print(f"📌 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES FOR COT")
print("=" * 60)
print()
print("1. CoT works best with temperature 0.0")
print("2. Always explicitly request step-by-step reasoning")
print("3. Verify critical outputs - don't trust blindly")
print("4. Use CoT for accuracy, skip for speed")
print("5. Combine CoT with structured output for best results")
print()

CHAIN-OF-THOUGHT BEST PRACTICES

📌 When to Use CoT
------------------------------------------------------------
  ✓ Multi-step problems (math, logic, planning)
  ✓ When accuracy is critical
  ✓ When you need to verify reasoning
  ✓ When the problem is complex or tricky
  ✓ When debugging incorrect answers
  ✗ Simple factual questions
  ✗ Single-step tasks
  ✗ When speed is more important than accuracy

📌 Prompting Techniques
------------------------------------------------------------
  ✓ Use explicit phrases: 'Let's think step-by-step'
  ✓ Guide the structure: numbered steps, categories
  ✓ Ask for verification: 'Now check your work'
  ✓ Request final answer separately: 'Therefore, the answer is...'
  ✓ Use temperature 0.0 for consistency
  ✗ Don't assume CoT will happen automatically
  ✗ Don't leave reasoning structure vague

📌 Parsing CoT Outputs
------------------------------------------------------------
  ✓ Look for clear step markers (1., 2., 3.)
  ✓ Extract final answer explici

---
## ✅ Notebook 05 Complete!

### Summary

You've mastered Chain-of-Thought prompting! You now know:
- ✅ What CoT is and why it works
- ✅ Multiple CoT prompting patterns
- ✅ When to use (and not use) CoT
- ✅ How to debug reasoning with CoT
- ✅ Advanced CoT techniques
- ✅ Production best practices

In [7]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on Chain-of-Thought
# ============================================================================

reflection = """
### 1. How has CoT changed your approach to complex problems?

CoT has made me approach problems more systematically. Instead of rushing to an answer, I break the problem into smaller logical steps, which helps catch mistakes early and improves accuracy, especially for multi-step reasoning tasks.

### 2. When will you use CoT vs regular prompting?

I will use CoT whenever:
- The problem requires multiple logical or arithmetic steps
- High accuracy is critical
- The solution benefits from explicit reasoning that can be verified

Regular prompting is sufficient when:
- The task is simple or factual
- Step-by-step reasoning is unnecessary
- Speed is more important than transparency

### 3. What's your confidence in using CoT? (1-5)

**Confidence:** 4/5

My confidence would increase with more practice on edge cases, handling ambiguous inputs, and integrating validation in production pipelines.

### 4. Most surprising finding about CoT?

The biggest “aha!” moment was realizing that CoT can prevent intuitive errors, even on problems that seem simple (like the bat and ball problem). Explicit reasoning forces the model to slow down and avoid traps.

### 5. How will you use CoT in your projects?

- Automating math problem solutions for tutoring or homework checking
- Structuring complex data extraction pipelines
- Debugging AI-generated outputs for accuracy
- Step-by-step reasoning in decision support tools

### 6. CoT limitations you discovered?

- Longer CoT outputs can be truncated or hard to read
- CoT may increase token usage, which can raise API costs
- For very creative tasks, strict CoT can sometimes limit flexibility
- It’s not a substitute for validation; models can still make arithmetic or logic errors

### 7. Key takeaway from this notebook?

CoT is a powerful tool for structured reasoning and transparency. It works best when combined with verification or schema validation, and it improves confidence in multi-step problem-solving tasks.
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="05",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 05 COMPLETE!")
print("=" * 60)
print()
print("Progress: [████████████████░░░░] 63% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs")
print("✓ Notebook 05: Chain of Thought ← YOU ARE HERE")
print("○ Notebook 06: Model Comparison")
print("○ Notebook 07: MCP Introduction")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/06_model_comparison.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. How has CoT changed your approach to complex problems?

CoT has made me approach problems more systematically. Instead of rushing to an answer, I break the problem into smaller logical steps, which helps catch mistakes early and improves accuracy, especially for multi-step reasoning tasks.

### 2. When will you use CoT vs regular prompting?

I will use CoT whenever:
- The problem requires multiple logical or arithmetic steps
- High accuracy is critical
- The solution benefits from explicit reasoning that can be verified

Regular prompting is sufficient when:
- The task is simple or factual
- Step-by-step reasoning is unnecessary
- Speed is more important than transparency

### 3. What's your confidence in using CoT? (1-5)

**Confidence:** 4/5

My confidence would increase with more practice on edge cases, handling ambiguous inputs, and integrating validation in production pipelines.

### 4. Most surprising finding about CoT?

The biggest “aha!” mome